# SegFormer fine-tune on `training/export.zip`

Fine-tunes `nvidia/mit-b0` on the tissue annotations exported from the web UI.  Output goes to `models/segformer/`, which the backend mounts read-only at `/models/segformer` and picks up automatically.

Tested with Python 3.11 + torch 2.2 + transformers 4.45.  On Apple Silicon, training uses MPS (~10 min for 30 images, 50 epochs); a CUDA GPU is faster.

## Workflow
1. Download `training/export.zip` from the dashboard.
2. Put it next to this notebook and run every cell.
3. Copy the resulting `models/segformer/` directory into the repo's `models/segformer/`.
4. `docker compose restart backend` — the SegFormer panel now reports "checkpoint 検出".

In [ ]:
# !pip install -q "torch>=2.2" "transformers>=4.45" "safetensors>=0.4" datasets pillow numpy scikit-learn

In [ ]:
import json, zipfile, os, random
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from transformers import (
    SegformerForSemanticSegmentation,
    SegformerImageProcessor,
)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

EXPORT_ZIP = Path("plants-research-training.zip")
WORK = Path("_training_work")
WORK.mkdir(exist_ok=True)
OUT = Path("../models/segformer")
OUT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(EXPORT_ZIP) as zf:
    zf.extractall(WORK)

classes = json.loads((WORK / "classes.json").read_text())
index = json.loads((WORK / "index.json").read_text())
class_keys = [c["key"] for c in classes["classes"]]  # 1..N
NUM_LABELS = len(class_keys) + 1  # +1 for background
id2label = {0: "background", **{c["index"]: c["key"] for c in classes["classes"]}}
label2id = {v: k for k, v in id2label.items()}
print(f"{len(index['images'])} images, {NUM_LABELS} labels")

In [ ]:
class TissueDataset(Dataset):
    def __init__(self, rows, processor, size=512):
        self.rows = rows
        self.processor = processor
        self.size = size

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        img = Image.open(WORK / row["image_path"]).convert("RGB").resize((self.size, self.size), Image.BILINEAR)
        mask = Image.open(WORK / row["mask_path"]).resize((self.size, self.size), Image.NEAREST)
        enc = self.processor(images=img, segmentation_maps=mask, return_tensors="pt")
        return {k: v.squeeze(0) for k, v in enc.items()}

processor = SegformerImageProcessor.from_pretrained("nvidia/mit-b0")
rows = [r for r in index["images"] if r["annotation_count"] > 0]
random.shuffle(rows)
split = max(1, int(0.8 * len(rows)))
train_rows, val_rows = rows[:split], rows[split:] or rows[:1]
train_ds = TissueDataset(train_rows, processor)
val_ds = TissueDataset(val_rows, processor)
print(f"train={len(train_ds)} val={len(val_ds)}")

In [ ]:
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/mit-b0",
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
).to(device)
optim = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)
train_dl = DataLoader(train_ds, batch_size=2, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=2)
print(f"device={device}")

In [ ]:
EPOCHS = 50
model.train()
for epoch in range(EPOCHS):
    losses = []
    for batch in train_dl:
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(**batch)
        out.loss.backward()
        optim.step()
        optim.zero_grad()
        losses.append(out.loss.item())
    print(f"epoch {epoch+1:02d} loss={np.mean(losses):.4f}")

In [ ]:
# quick val: mean pixel accuracy
model.eval()
with torch.no_grad():
    correct = total = 0
    for batch in val_dl:
        batch = {k: v.to(device) for k, v in batch.items()}
        logits = model(pixel_values=batch["pixel_values"]).logits
        logits = torch.nn.functional.interpolate(logits, size=batch["labels"].shape[-2:], mode="bilinear", align_corners=False)
        pred = logits.argmax(dim=1)
        mask = batch["labels"] != 255
        correct += (pred[mask] == batch["labels"][mask]).sum().item()
        total += mask.sum().item()
print(f"val pixel acc = {correct / max(total, 1):.3f}")

In [ ]:
# Save checkpoint in the layout the backend expects.
model.save_pretrained(OUT, safe_serialization=True)
processor.save_pretrained(OUT)
print(f"saved to {OUT.resolve()}")
for p in sorted(OUT.iterdir()):
    print("  ", p.name)